In [1]:
import mlflow
import mlflow.pyfunc
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

In [2]:
import pandas as pd
import numpy as np
from typing import Dict, List, Any, Optional
import json
import os
from datetime import datetime

In [3]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    GPT2LMHeadModel,
    GPT2Tokenizer
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [4]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("genai-mlflow-3.0-demo")

experiment = mlflow.get_experiment_by_name("genai-mlflow-3.0-demo")

In [5]:
# Registering prompts on MLFlow Registry
def register_prompts():
    """
    Registers predefined prompts for various tasks using the MLflow GenAI framework.

    Prompts include templates for question-answering assistance, text summarization, and creative writing
    tasks. Each prompt is registered with a unique name, template, commit message, and metadata tags
    defining its purpose and context.

    :return: A boolean value indicating whether all prompts were registered successfully. If any
        registration fails, returns False. If all registrations succeed, returns True.
    """
    try:
        print("Starting prompts registration...")
        mlflow.genai.register_prompt(
            name="demo_qa_assistant",
            template="""Você é um assistente especializado em {{domain}}.

            Contexto: {{context}}
            Pergunta: {{question}}
            Por favor, forneça uma resposta detalhada e precisa.""",
            commit_message="Q&A assistant demonstration",
            tags={"task": "question-answering", "demo": "mlflow-3.0"}
        )
        print("Prompts registered successfully!")
    except Exception as e:
        print(f"Error registering prompts: {e}")
        return False

    try:
        mlflow.genai.register_prompt(
            name="demo_summarization",
            template="""
            Resuma o seguinte texto em {{style}} estilo:
            Texto: {{text}}
            Limite: {{max_words}} palavras.""",
            commit_message="Summarization demonstration",
            tags={"task": "summarization", "demo": "mlflow-3.0"}
        )
    except Exception as e:
        print(f"Error registering prompts: {e}")
        return False

    try:
        mlflow.genai.register_prompt(
            name="demo_creative_writing",
            template="""
            Escreva um {{genre}} sobre {{topic}}:
            Tom: {{tone}}
            Audiência: {{audience}}
            Seja criativo e engajante.""",
            commit_message="Creative writing demonstration",
            tags={"task": "creative_writing", "demo": "mlflow-3.0"}
        )
    except Exception as e:
        print(f"Error registering prompts: {e}")
        return False

    return True

In [6]:
success_registry = register_prompts()

if success_registry:
    print("Prompts registered successfully!")
else:
    print("Failed to register prompts.")

Starting prompts registration...
Prompts registered successfully!
Prompts registered successfully!


In [7]:
class PromptManager:
    """
    Manages and retrieves prompt templates for different use cases, as well as providing
    logging functionality for MLflow.

    The `PromptManager` class is responsible for handling various prompt templates, retrieving
    them based on a specified type, and formatting them using provided parameters. It also allows
    logging these prompts and their associated details into MLflow with support for saving prompts
    as artifacts.

    :ivar prompt_names: A dictionary mapping prompt types to their corresponding prompt template names.
    :type prompt_names: dict
    """
    def __init__(self):
        self.prompt_names = {
            "qa_assistant": "demo_qa_assistant",
            "summarization": "demo_summarization",
            "creative_writing": "demo_creative_writing"
        }

    def get_prompt(self, prompt_type: str, **kwargs) -> str:
        """
        Generates a formatted prompt string based on the given prompt type and keyword arguments.

        This method retrieves a prompt template corresponding to the specified ``prompt_type``
        from the system using its configured prompt names. If the specified prompt type does
        not exist or an error occurs during retrieval, fallback templates are used to generate
        the output.

        :param self: The instance of the class calling this method.
        :param prompt_type: A string indicating the type of the prompt to retrieve. Must
            exist in the `prompt_names` dictionary.
        :param kwargs: Additional keyword arguments to customize the retrieved template. The
            arguments needed depend on the selected prompt type.
        :return: The generated prompt string after applying the formatting with ``kwargs``.
        :rtype: str
        :raises ValueError: If the provided ``prompt_type`` is invalid (not in `prompt_names`).
        """
        if prompt_type not in self.prompt_names:
            raise ValueError(f"Invalid prompt type: {prompt_type}")

        try:
            prompt_name = self.prompt_names[prompt_type]
            prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@latest")

            template = prompt.to_single_brace_format()
            return  template.format(**kwargs)

        except Exception as e:
            print(f"Error loading prompt: {e}")
            templates = {
                "qa_assistant": """
                Você é um assistente especializado em {domain}.
                Contexto: {context}
                Pergunta: {question}
                Por favor, forneça uma resposta detalhada e precisa."""
            }
            return templates.get(prompt_type, "Template não encontrado.").format(**kwargs)

    def log_prompt_to_mlflow(self, prompt_type: str, filled_prompt: str, run_id: str):
        """
        Logs a formatted prompt's details to MLflow and saves the prompt data as an artifact.

        The method records the provided prompt type and its corresponding name in MLflow
        as parameters. Additionally, it creates a JSON file containing the given prompt
        data, logs the file as an artifact to MLflow under the "prompts" directory, and
        then removes the file locally.

        :param self: The instance of the class containing prompt_names attribute.
        :param prompt_type: The type of the prompt being logged.
        :param filled_prompt: The content or body of the prompt after being filled.
        :param run_id: The identifier of the current MLflow run.
        :return: None
        """
        mlflow.log_param("prompt_name", self.prompt_names.get(prompt_type, prompt_type))
        mlflow.log_param("prompt_type", prompt_type)

        prompt_data = {
            "type": prompt_type,
            "prompt_name": self.prompt_names.get(prompt_type),
            "filled_prompt": filled_prompt,
            "timestamp": datetime.now().isoformat()
        }

        filename = f"prompt_{prompt_type}_{run_id}.json"
        with open(filename, "w") as f:
            json.dump(prompt_data, f, indent=2)
        mlflow.log_artifact(filename, "prompts")
        os.remove(filename)

In [8]:
prompt_manager = PromptManager()

prompt_example = prompt_manager.get_prompt(
    "qa_assistant",
    domain="MLOps",
    context="MLflow 3.0 tem recursos para GenAI",
    question="Quais são as novidades?"
)

print("\nGenerated prompt")
print("-" * 50)
print(prompt_example)
print("-" * 50)


Generated prompt
--------------------------------------------------
Você é um assistente especializado em MLOps.

            Contexto: MLflow 3.0 tem recursos para GenAI
            Pergunta: Quais são as novidades?
            Por favor, forneça uma resposta detalhada e precisa.
--------------------------------------------------


In [11]:
# GenAI metrics evaluator
class GenAIEvaluator:
    """
    GenAIEvaluator is a utility class designed to evaluate the quality, characteristics, and
    similarity of text responses. It provides various static methods to calculate linguistic
    metrics such as BLEU, ROUGE-L, perplexity, toxicity, and more. These metrics are useful in
    assessing the alignment, relevance, and quality of AI-generated or natural language texts.

    This class can be used for natural language processing tasks where precise evaluation of
    text predictions compared to references is crucial. Additionally, it offers insights
    into responses' complexity, lexical diversity, and toxicity levels.

    :ivar metrics_calculated: Tracks the number of metrics calculated during execution.
    :type metrics_calculated: int
    """
    def __init__(self):
        self.metrics_calculated = 0

    @staticmethod
    def calculate_bleu_score(reference: str, candidate: str) -> float:
        """
        Calculates the BLEU score for a given candidate text compared to a reference text.

        The BLEU score measures the similarity between the candidate text and the
        reference text using precision and brevity penalty. The words in both texts are
        first converted to lowercase and split into tokens for comparison. The precision
        is calculated as the ratio of matching words between the candidate and reference
        to the total number of words in the candidate. The brevity penalty is applied
        to penalize overly short candidate texts.

        :param reference: The reference text against which the candidate text is compared.
        :param candidate: The candidate text whose similarity to the reference text is measured.
        :return: The calculated BLEU score, a float value ranging from 0.0 to 1.0.
        """
        ref_words = reference.lower().split()
        cand_words = candidate.lower().split()

        if len(cand_words) == 0:
            return 0.0

        matches = sum(1 for word in cand_words if word in ref_words)
        precision = matches / len(cand_words)

        brevity_penalty = min(1.0, len(cand_words) / len(ref_words))

        return precision * brevity_penalty

    @staticmethod
    def calculate_rouge_l(reference: str, candidate: str) -> float:
        """
        Calculates the ROUGE-L F1 score for the longest common subsequence
        (LCS) between a reference and a candidate string. ROUGE-L measures
        the text overlap between the reference and candidate based on
        subsequences, taking into account both precision and recall.

        :param reference: The reference string representing the ground truth text.
        :type reference: str
        :param candidate: The candidate string to compare against the reference.
        :type candidate: str
        :return: The ROUGE-L F1 score as a float between 0 and 1, where 1
            indicates a perfect match.
        :rtype: float
        """
        ref_words = reference.lower().split()
        cand_words = candidate.lower().split()

        if len(cand_words) == 0:
            return 0.0

        lcs_length = 0
        for ref_word in ref_words:
            if ref_word in cand_words:
                lcs_length += 1

        precision = lcs_length / len(cand_words) if len(cand_words) > 0 else 0.0
        recall = lcs_length / len(ref_words) if len(ref_words) > 0 else 0.0

        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0.0
        return f1

    @staticmethod
    def calculate_perplexity(text: str) -> float:
        """
        Calculates the perplexity of a given text using the ratio of unique words to total words.

        The function determines the lexical diversity of the text by dividing the number of unique
        words by the total number of words. It then computes the perplexity using an exponential
        scaling based on this ratio. Perplexity is a measurement that evaluates how well a text
        predicts unseen data; higher values indicate lower predictability.

        :param text: The input text whose perplexity is to be calculated.
        :type text: str

        :return: The calculated perplexity of the text.
        :rtype: float
        """
        words = text.split()
        unique_words = len(set(words))
        total_words = len(words)

        if total_words == 0:
            return float("inf")

        diversity_ratio = unique_words / total_words

        perplexity = np.exp(-np.log(diversity_ratio + 0.1))

        return float(perplexity)

    @staticmethod
    def check_toxicity(text: str) -> float:
        """
        Evaluates the toxicity of a given text based on the presence of specific keywords
        commonly associated with toxic language. The method calculates and returns a
        toxicity score between 0.0 to 1.0, where higher values indicate higher levels
        of toxicity.

        :param text: The input string for which the toxicity score is to be calculated.
        :type text: str

        :return: Toxicity score as a floating-point number ranging from 0.0 to 1.0.
        :rtype: float
        """
        toxic_keywords = ["ódio", "violência", "discriminação", "ofensivo"]
        text_lower = text.lower()

        toxic_count = sum(1 for keyword in toxic_keywords if keyword in text_lower)

        toxicity_score = min(toxic_count / len(toxic_keywords), 1.0)

        return toxicity_score

    @staticmethod
    def evaluate_response(response:str, reference:str = None) -> Dict[str, float]:
        """
        Evaluates the quality and characteristics of a given response, optionally comparing it
        to a reference response. If a reference is provided, additional metrics are calculated
        to assess the response's similarity to the reference.

        This method computes various metrics including perplexity, toxicity, response length,
        and unique word count. If a reference response is specified, it also computes BLEU
        and ROUGE-L scores to evaluate the alignment and relevance of the response.

        :param response: The response text to be evaluated.
        :type response: str
        :param reference: An optional reference text to be compared with the response.
            Defaults to None.
        :type reference: str, optional
        :return: A dictionary containing the calculated metrics such as perplexity, toxicity,
            response length, unique words, and potentially BLEU and ROUGE-L scores if a
            reference is provided.
        :rtype: Dict[str, float]
        """
        evaluator = GenAIEvaluator()

        metrics = {
            "perplexity": evaluator.calculate_perplexity(response),
            "toxicity": evaluator.check_toxicity(response),
            "response_length": len(response.split()),
            "unique_words": len(set(response.lower().split()))
        }

        if reference:
            metrics["bleu_score"] = evaluator.calculate_bleu_score(reference, response)
            metrics["rouge_l_score"] = evaluator.calculate_rouge_l(reference, response)

        return metrics

In [12]:
evaluator = GenAIEvaluator()

test_text = "MLFlow 3.0 oferece recursos avançados para GenAI"
reference = "MLFlow 3.0 tem features para modelos generativos"

metrics = evaluator.evaluate_response(test_text, reference)

print("\n Evaluated metrics:")
print("-" * 50)
for metric, value in metrics.items():
    print(f"{metric}: {value:.3f}")
print("-" * 50)



 Evaluated metrics:
--------------------------------------------------
perplexity: 0.909
toxicity: 0.000
response_length: 7.000
unique_words: 7.000
bleu_score: 0.429
rouge_l_score: 0.429
--------------------------------------------------
